In [1]:
from pydantic import BaseModel
from typing import List


class PersonInfo(BaseModel):
    name: str
    age: int
    company: str
    skills: List[str]

In [2]:
person = PersonInfo(
    name="Yash",
    age=22,
    company="Kombee",
    skills=["Python", "FastAPI", "LangChain"]
)

print(person)

name='Yash' age=22 company='Kombee' skills=['Python', 'FastAPI', 'LangChain']


In [4]:
person = PersonInfo(
    name="Yash",
    age="Hello",
    company="Kombee",
    skills=["Python", "FastAPI", "LangChain"]
)

print(person)

ValidationError: 1 validation error for PersonInfo
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='Hello', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing

In [5]:
from google import genai
from google.genai import types
import os
from dotenv import load_dotenv

load_dotenv()

True

In [6]:
client = genai.Client(
    api_key= os.getenv("GEMINI_API_KEY")
)

In [7]:
text = """
Rahul is 25 years old and works at ABC Technologies.
He is a Python and FastAPI developer.
He also has experience with artificial intelligence.
"""

In [8]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=f"""
Extract the person's information from the following text.

TEXT:
{text}
""",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=PersonInfo
    )
)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


In [9]:
print(response.text)

{"name":"Rahul","age":25,"company":"ABC Technologies","skills":["Python","FastAPI","artificial intelligence"]}


In [10]:
person = PersonInfo.model_validate_json(response.text)

print(person)

name='Rahul' age=25 company='ABC Technologies' skills=['Python', 'FastAPI', 'artificial intelligence']


In [11]:
print(person.name)
print(person.age)
print(person.company)
print(person.skills)

Rahul
25
ABC Technologies
['Python', 'FastAPI', 'artificial intelligence']


In [12]:
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    
    weather_data = {
        "surat": "32°C, Sunny",
        "ahmedabad": "34°C, Clear",
        "mumbai": "29°C, Cloudy"
    }

    return weather_data.get(
        city.lower(),
        f"Weather data not available for {city}"
    )

In [13]:
print(get_weather("Surat"))
print(get_weather("Mumbai"))
print(get_weather("Delhi"))

32°C, Sunny
29°C, Cloudy
Weather data not available for Delhi


In [14]:
weather_tool = types.FunctionDeclaration(
    name="get_weather",
    description="Get the current weather information for a city.",
    parameters=types.Schema(
        type="OBJECT",
        properties={
            "city": types.Schema(
                type="STRING",
                description="Name of the city"
            )
        },
        required=["city"]
    )
)

In [15]:
tools = types.Tool(
    function_declarations=[weather_tool]
)

In [17]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="What is the weather in Surat?",
    config=types.GenerateContentConfig(
        tools=[tools]
    )
)

In [18]:
for part in response.candidates[0].content.parts:
    if part.function_call:
        print("Function:", part.function_call.name)
        print("Arguments:", part.function_call.args)

Function: get_weather
Arguments: {'city': 'Surat'}


In [19]:
for part in response.candidates[0].content.parts:

    if part.function_call:

        function_name = part.function_call.name
        arguments = part.function_call.args

        if function_name == "get_weather":
            result = get_weather(arguments["city"])

            print("Tool executed!")
            print("Tool result:", result)

Tool executed!
Tool result: 32°C, Sunny


In [21]:
tool_result = None

for part in response.candidates[0].content.parts:
    if part.function_call:
        function_name = part.function_call.name
        arguments = part.function_call.args

        if function_name == "get_weather":
            tool_result = get_weather(arguments["city"])

print("Tool result:", tool_result)

Tool result: 32°C, Sunny


In [22]:
final_response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=f"""
    The user asked: What is the weather in Surat?

    The weather tool returned:
    {tool_result}

    Give the user a clear and natural final answer.
    """
)

print(final_response.text)

The weather in Surat is currently sunny with a temperature of 32°C.


In [23]:
final_text = """
Rahul Patel is 25 years old and works at ABC Technologies.
He is a Python and FastAPI developer.
His skills include artificial intelligence and machine learning.
"""

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=f"""
    Extract the person's information from this text.

    TEXT:
    {final_text}
    """,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=PersonInfo
    )
)

print(response.text)

{"name":"Rahul Patel","age":25,"company":"ABC Technologies","skills":["Python","FastAPI","artificial intelligence","machine learning"]}


In [24]:
person = PersonInfo.model_validate_json(response.text)

print("Validated successfully!")
print(person)

Validated successfully!
name='Rahul Patel' age=25 company='ABC Technologies' skills=['Python', 'FastAPI', 'artificial intelligence', 'machine learning']


In [25]:
def run_weather_agent(question):

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=question,
        config=types.GenerateContentConfig(
            tools=[tools]
        )
    )

    for part in response.candidates[0].content.parts:

        if part.function_call:

            function_name = part.function_call.name
            arguments = part.function_call.args

            if function_name == "get_weather":
                result = get_weather(arguments["city"])

                final_response = client.models.generate_content(
                    model="gemini-3.6-flash",
                    contents=f"""
                    User question:
                    {question}

                    Tool result:
                    {result}

                    Give a clear final answer to the user.
                    """
                )

                return final_response.text

    return response.text

In [26]:
print(run_weather_agent("What is the weather in Mumbai?"))

The current weather in Mumbai is 29°C and cloudy.


In [27]:
print(run_weather_agent("What is the weather in Ahmedabad?"))

The weather in Ahmedabad is currently **34°C** and **clear**.
